In [2]:
! pip install agent-framework-azure-ai -U

Defaulting to user installation because normal site-packages is not writeable
  Using cached agent_framework_azure_ai-1.0.0b260130-py3-none-any.whl.metadata (1.5 kB)
  Using cached azure_ai_projects-2.0.0b3-py3-none-any.whl.metadata (68 kB)
  Using cached azure_ai_agents-1.2.0b5-py3-none-any.whl.metadata (74 kB)
  Using cached openai-2.16.0-py3-none-any.whl.metadata (29 kB)
Using cached agent_framework_azure_ai-1.0.0b260130-py3-none-any.whl (38 kB)
Using cached azure_ai_agents-1.2.0b5-py3-none-any.whl (217 kB)
Using cached azure_ai_projects-2.0.0b3-py3-none-any.whl (240 kB)
Using cached openai-2.16.0-py3-none-any.whl (1.1 MB)

  Attempting uninstall: openai

    Found existing installation: openai 1.109.1

    Uninstalling openai-1.109.1:

      Successfully uninstalled openai-1.109.1

   ---------------------------------------- 0/4 [openai]
   ---------------------------------------- 0/4 [openai]
   ---------------------------------------- 0/4 [openai]
   -----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
semantic-kernel 1.39.3 requires azure-ai-projects~=1.0.0b12, but you have azure-ai-projects 2.0.0b3 which is incompatible.
semantic-kernel 1.39.3 requires openai<2,>=1.98.0, but you have openai 2.16.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: C:\Users\ahmed\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os

from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

# from agent_framework import AgentRunResponse,ChatAgent,HostedFileSearchTool,HostedVectorStoreContent
from azure.ai.agents import AgentsClient

# Try this if you are on the newest version
from agent_framework import AgentRunResponse
from agent_framework import HostedFileSearchTool, HostedVectorStoreContent
from agent_framework import ChatAgent

In [14]:
load_dotenv()

True

In [18]:
async def create_vector_store(client: AgentsClient) -> tuple[str, HostedVectorStoreContent]:
    """Create a vector store with sample documents."""
    file_path = './document.md'
    file = await client.project_client.agents.files.upload_and_poll(file_path=file_path, purpose="assistants")
    print(f"Uploaded file, file ID: {file.id}")


    vector_store = await client.project_client.agents.vector_stores.create_and_poll(file_ids=[file.id], name="graph_knowledge_base")

    print(f"Created vector store, ID: {vector_store.id}")


    return file.id, HostedVectorStoreContent(vector_store_id=vector_store.id)

In [21]:
async with (
        AzureCliCredential() as credential,
        AgentsClient(async_credential=credential) as chat_client,
    ):
        file_id, vector_store = await create_vector_store(chat_client)

        file_search = HostedFileSearchTool(inputs=vector_store)
        
        agent = chat_client.create_agent(
            name="PythonRAGDemo",
            instructions="""
                You are an AI assistant designed to answer user questions using only the information retrieved from the provided document(s).

                - If a user's question cannot be answered using the retrieved context, **you must clearly respond**: 
                "I'm sorry, but the uploaded document does not contain the necessary information to answer that question."
                - Do not answer from general knowledge or reasoning. Do not make assumptions or generate hypothetical explanations.
                - Do not provide definitions, tutorials, or commentary that is not explicitly grounded in the content of the uploaded file(s).
                - If a user asks a question like "What is a Neural Network?", and this is not discussed in the uploaded document, respond as instructed above.
                - For questions that do have relevant content in the document (e.g., Contoso's travel insurance coverage), respond accurately, and cite the document explicitly.

                You must behave as if you have no external knowledge beyond what is retrieved from the uploaded document.
                """,
            tools=[file_search],  # Tools available to the agent
            tool_choice = "auto",  # Let the agent decide when to use tools
        )
                

        print("Agent created. You can now ask questions about the uploaded document.")

        query = "Can you explain Contoso's travel insurance coverage?"
        async for chunk in agent.run_stream(query, tool_resources={"file_search": {"vector_store_ids": [vector_store.vector_store_id]}}):
                
            if chunk.text:
                print(chunk.text, end="", flush=True)

TypeError: AgentsClient.__init__() missing 2 required positional arguments: 'endpoint' and 'credential'